In [1]:
import pandas as pd
import os

print("--- KHỐI 1: TẠO FILE TỪ ĐIỂN FEATURE (FEATURE CATALOG) ---")

# Định nghĩa danh sách các Feature
catalog_data = [
    # Nhóm 1: Temporal Features (Thời gian)
    ['Temporal', 'pickup_hour', 'tpep_pickup_datetime.dt.hour', 'Xác định thời điểm di chuyển (Sáng/Chiều/Tối)'],
    ['Temporal', 'pickup_day_of_week', 'tpep_pickup_datetime.dt.dayofweek', 'Ngày trong tuần (0=Thứ 2, 6=CN)'],
    ['Temporal', 'is_weekend', '1 nếu day_of_week >= 5, ngược lại 0', 'Phân biệt khách hàng đi làm (Weekday) và đi chơi (Weekend)'],
    ['Temporal', 'is_rush_hour', '1 nếu giờ thuộc [7,8,9,17,18,19]', 'Tệp khách hàng Commuter (Đi làm/Tan tầm)'],
    ['Temporal', 'is_night_time', '1 nếu giờ >= 22 hoặc <= 4', 'Tệp khách hàng Nightlife (Đi chơi đêm/Bar pub)'],
    
    # Nhóm 2: Trip Features (Chuyến đi)
    ['Trip', 'trip_distance', 'Lấy giá trị gốc (Miles)', 'Phân biệt khách đi nội đô ngắn và khách đi tỉnh/đường dài'],
    ['Trip', 'trip_duration_mins', '(Dropoff - Pickup) tính bằng Phút', 'Mức độ kiên nhẫn/Thời gian ngồi trên xe'],
    ['Trip', 'fare_per_mile', 'fare_amount / trip_distance', 'Đo độ đắt đỏ. Đi ngắn nội đô thường có giá/dặm rất cao do kẹt xe'],
    ['Trip', 'tip_pct', '(tip_amount / fare_amount) * 100', 'Đo độ hào phóng. Phân biệt tệp Khách VIP vs Khách Bình dân'],
    ['Trip', 'passenger_count', 'Lấy giá trị gốc', 'Phân biệt Solo Rider (1 người) và Group Rider (Nhiều người)'],
    
    # Nhóm 3: Geographic Features (Địa lý)
    ['Geographic', 'pu_Borough', 'Join từ taxi_zone_lookup', 'Quận đón khách trọng điểm'],
    ['Geographic', 'do_Borough', 'Join từ taxi_zone_lookup', 'Quận trả khách trọng điểm'],
    ['Geographic', 'is_airport_trip', '1 nếu RatecodeID=2,3 hoặc Zone chứa Airport', 'Tệp khách sân bay giá trị cao, đi xa, ít nhạy cảm về giá'],
    ['Geographic', 'is_intra_manhattan', '1 nếu đón VÀ trả đều ở Manhattan', 'Tệp khách di chuyển nội khu trung tâm (thường đi rất ngắn)']
]

# Tạo DataFrame
df_catalog = pd.DataFrame(catalog_data, columns=['Feature Group', 'Feature Name', 'Calculation Logic', 'Business / Segmentation Rationale'])

# Lưu ra file Excel trong thư mục docs
out_path = r'D:\Intern VSF\GSM-promotion-experimentation\docs\feature_catalog.xlsx'
df_catalog.to_excel(out_path, index=False, sheet_name='Feature Catalog')

print(f"✅ Đã tạo thành công file: {out_path}")
display(df_catalog)


--- KHỐI 1: TẠO FILE TỪ ĐIỂN FEATURE (FEATURE CATALOG) ---
✅ Đã tạo thành công file: D:\Intern VSF\GSM-promotion-experimentation\docs\feature_catalog.xlsx


,Feature Group,Feature Name,Calculation Logic,Business / Segmentation Rationale
0,Temporal,pickup_hour,tpep_pickup_datetime.dt.hour,Xác định thời điểm di chuyển (Sáng/Chiều/Tối)
1,Temporal,pickup_day_of_week,tpep_pickup_datetime.dt.dayofweek,"Ngày trong tuần (0=Thứ 2, 6=CN)"
2,Temporal,is_weekend,"1 nếu day_of_week >= 5, ngược lại 0",Phân biệt khách hàng đi làm (Weekday) và đi ch...
3,Temporal,is_rush_hour,"1 nếu giờ thuộc [7,8,9,17,18,19]",Tệp khách hàng Commuter (Đi làm/Tan tầm)
4,Temporal,is_night_time,1 nếu giờ >= 22 hoặc <= 4,Tệp khách hàng Nightlife (Đi chơi đêm/Bar pub)
5,Trip,trip_distance,Lấy giá trị gốc (Miles),Phân biệt khách đi nội đô ngắn và khách đi tỉn...
6,Trip,trip_duration_mins,(Dropoff - Pickup) tính bằng Phút,Mức độ kiên nhẫn/Thời gian ngồi trên xe
7,Trip,fare_per_mile,fare_amount / trip_distance,Đo độ đắt đỏ. Đi ngắn nội đô thường có giá/dặm...
8,Trip,tip_pct,(tip_amount / fare_amount) * 100,Đo độ hào phóng. Phân biệt tệp Khách VIP vs Kh...
9,Trip,passenger_count,Lấy giá trị gốc,Phân biệt Solo Rider (1 người) và Group Rider ...


In [2]:
import numpy as np
import pandas as pd

print("--- KHỐI 2: TÍNH TOÁN VÀ XUẤT FEATURE STORE V1 ---")

# 1. Load dữ liệu đã làm sạch
data_path = r'D:\Intern VSF\GSM-promotion-experimentation\data\yellow_tripdata_2026-01_CLEANED.parquet'
print("Đang load dữ liệu...")
df = pd.read_parquet(data_path)

# --- TÍNH TOÁN TEMPORAL FEATURES ---
print("Đang tính toán nhóm Temporal...")
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek
df['is_weekend'] = (df['pickup_day_of_week'] >= 5).astype(int)
df['is_rush_hour'] = df['pickup_hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df['is_night_time'] = ((df['pickup_hour'] >= 22) | (df['pickup_hour'] <= 4)).astype(int)

# --- TÍNH TOÁN TRIP FEATURES ---
print("Đang tính toán nhóm Trip...")
df['trip_duration_mins'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
df['fare_per_mile'] = np.where(df['trip_distance'] > 0, df['fare_amount'] / df['trip_distance'], 0)
df['tip_pct'] = np.where((df['fare_amount'] > 0) & (df['tip_amount'] >= 0), 
                         (df['tip_amount'] / df['fare_amount']) * 100, 0)

# --- TÍNH TOÁN GEOGRAPHIC FEATURES ---
print("Đang tính toán nhóm Geographic...")
is_airport_ratecode = df['RatecodeID'].isin([2.0, 3.0])
is_airport_zone = df['pu_Zone'].str.contains('Airport', na=False, case=False) | df['do_Zone'].str.contains('Airport', na=False, case=False)
df['is_airport_trip'] = (is_airport_ratecode | is_airport_zone).astype(int)
df['is_intra_manhattan'] = ((df['pu_Borough'] == 'Manhattan') & (df['do_Borough'] == 'Manhattan')).astype(int)

# 2. Chọn lọc các cột cần thiết cho Segmentation (Feature Store)
feature_cols = [
    'tpep_pickup_datetime', 
    'trip_distance', 'trip_duration_mins', 'fare_amount', 'fare_per_mile', 'tip_pct', 'passenger_count', 'payment_type',
    'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'is_rush_hour', 'is_night_time',
    'pu_Borough', 'do_Borough', 'is_airport_trip', 'is_intra_manhattan'
]

# Chỉ lấy những cuốc xe có logic hợp lệ để tuần sau train model không bị lỗi
feature_store = df[
    (df['trip_distance'] > 0) & 
    (df['trip_duration_mins'] >= 1) & (df['trip_duration_mins'] <= 180) &
    (df['fare_amount'] > 0)
][feature_cols].copy()

# 3. Lưu ra file CSV
out_csv_path = r'D:\Intern VSF\GSM-promotion-experimentation\data\feature_store_v1.csv'
print(f"Đang lưu file Feature Store ({len(feature_store):,} dòng)...")
feature_store.to_csv(out_csv_path, index=False)
print(f"✅ Đã xuất thành công: {out_csv_path}")


--- KHỐI 2: TÍNH TOÁN VÀ XUẤT FEATURE STORE V1 ---
Đang load dữ liệu...
Đang tính toán nhóm Temporal...
Đang tính toán nhóm Trip...
Đang tính toán nhóm Geographic...
Đang lưu file Feature Store (3,487,453 dòng)...
✅ Đã xuất thành công: D:\Intern VSF\GSM-promotion-experimentation\data\feature_store_v1.csv
